# Salting OOM
----------------
- Sat we have a df with user_id and purchase_amount , say 80% of the user_id comes from A , we want to apply a groupBy on User_id and sum of purchase_amount, it'll create partitions for each user_id all the user_ids past A are fine , the user_id A is soo big.
- Say the executor that takes the user_id A partion has 1GB memeory and the partion is 1.5GB memory, will give us Driver Out of memory error, making it impossible to process, to process must go in the memory.
- We need to add a new Salt columns i.e. some numberrs to brerak the partition into smaller partitions , it'll be at a random range.

# No Salting
--------------

In [0]:
from pyspark.sql.functions import col

In [0]:

### Example DataFrame with skewed user_id distribution
data = [("A", 10)] * 80000000 + [("B", 20)] * 1000 + [("C", 30)] * 1000
df = spark.createDataFrame(data, ["user_id", "purchase_amount"])

### Group by user_id and sum purchase_amount (no salting)
df_grouped = df.groupBy("user_id").sum("purchase_amount")

display(df_grouped)

# With Salting
--------------------

In [0]:
### Add salt column to break up large 'A' partition
num_salts = 10
df_salted = df.withColumn("salt",floor(rand() * num_salts))

### Group by user_id and salt, then sum purchase_amount
df_grouped = df_salted.groupBy("user_id", "salt").sum("purchase_amount")

### Aggregate back to user_id level
df_final = df_grouped.groupBy("user_id").sum("sum(purchase_amount)")

display(df_final)